# CheXpert Capstone — Unified Model Training (Google Colab)

Use this notebook to train **one** selected architecture with the same data split, label order, metrics and checkpoint format used by the frontend/backend.

Supported `MODEL_NAME` values: `vit`, `densenet`, `capsnet`, `gat`.

**Before starting:** upload `chexpert_capstone.zip` when prompted. The notebook extracts the source code, downloads/locates CheXpert using KaggleHub, then runs the shared training script. You need Kaggle access to download the dataset. Use a GPU runtime for practical training.


In [ ]:
from pathlib import Path
import zipfile

PROJECT_DIR = Path('/content/chexpert_capstone')

if not PROJECT_DIR.exists():
    from google.colab import files
    uploaded = files.upload()  # Select chexpert_capstone.zip
    zip_files = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_files:
        raise FileNotFoundError('Upload the chexpert_capstone.zip file.')
    with zipfile.ZipFile(zip_files[0]) as archive:
        archive.extractall('/content')

if not (PROJECT_DIR / 'backend' / 'train.py').exists():
    raise FileNotFoundError(f'Project source not found in {PROJECT_DIR}. Check the uploaded ZIP structure.')

import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Project:', PROJECT_DIR)


In [ ]:
# Colab normally includes torch and torchvision. Install the remaining packages only.
%pip -q install pandas scikit-learn pillow opencv-python-headless networkx tqdm kagglehub


In [ ]:
import torch
from backend.data import find_dataset_root
import kagglehub

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Download through KaggleHub. The first download may take some time.
download_path = kagglehub.dataset_download('ashery/chexpert')
DATA_ROOT = str(find_dataset_root(download_path))
print('CheXpert root:', DATA_ROOT)


## Choose the model and training settings

Start with a smaller subset to verify the data pipeline. After it works, increase the sample counts or set them to `0` to use the full CSV files. ViT usually needs a smaller batch size than the other models.


In [ ]:
# Change this line to: 'vit', 'densenet', 'capsnet', or 'gat'
MODEL_NAME = 'densenet'

EPOCHS = 3
BATCH_SIZE = 8
TRAIN_SAMPLES = 5000   # Set to 0 for the full training CSV
VALID_SAMPLES = 1000   # Set to 0 for the full validation CSV
NUM_WORKERS = 2
LEARNING_RATE = None   # None uses the model-specific default

# Save checkpoints in Google Drive so they can later be copied into the local app.
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/chexpert_capstone/checkpoints'
else:
    OUTPUT_DIR = str(PROJECT_DIR / 'checkpoints')

assert MODEL_NAME in {'vit', 'densenet', 'capsnet', 'gat'}
print('Selected model:', MODEL_NAME)
print('Output directory:', OUTPUT_DIR)


In [ ]:
import subprocess

command = [
    sys.executable, '-m', 'backend.train',
    '--model', MODEL_NAME,
    '--data-root', DATA_ROOT,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--max-train-samples', str(TRAIN_SAMPLES),
    '--max-valid-samples', str(VALID_SAMPLES),
    '--output-dir', OUTPUT_DIR,
]
if LEARNING_RATE is not None:
    command.extend(['--learning-rate', str(LEARNING_RATE)])

print('Running:', ' '.join(command))
subprocess.run(command, check=True)


In [ ]:
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
print('Saved files:')
for output_file in sorted(output_dir.glob(f'{MODEL_NAME}*')):
    print(output_file.name, f'({output_file.stat().st_size / (1024**2):.2f} MB)')


## Next steps

1. Repeat the model-selection and training cells for the other three model names.
2. Copy the four `.pt` checkpoints into the local project's `checkpoints/` folder.
3. Follow the root `README.md` to start the FastAPI backend and frontend locally.
4. Report metrics from the actual CheXpert validation data only. The tiny synthetic test results are code checks, not model performance results.

**Label policy reminder:** the shared baseline maps uncertain `-1` and missing `NaN` labels to `0`, matching the uploaded notebooks. Discuss this assumption in your methodology and consider evaluating alternative policies.
